# GuardNet — 01. Data Preprocessing

Loads the **NSL-KDD** dataset and prepares it for the Hybrid CNN-LSTM model:
log-transform skewed features → one-hot encode categoricals → standard-scale.

This notebook reuses the exact functions from `scripts/model_trainer.py`, so it
stays in sync with the trained model (no separate/duplicated logic).

In [ ]:
import os, sys
import numpy as np
import pandas as pd

BASE = os.path.dirname(os.getcwd())  # project root (parent of notebooks/)
sys.path.append(os.path.join(BASE, 'scripts'))
from model_trainer import load_dataset, COLUMNS, CATEGORICAL, SKEWED, TRAIN_PATH, TEST_PATH
print('Train file:', TRAIN_PATH)
print('Test  file:', TEST_PATH)

## 1. Inspect the raw NSL-KDD records
Each record has 41 features + a label. 3 features are categorical
(`protocol_type`, `service`, `flag`); the rest are numeric.

In [ ]:
raw = pd.read_csv(TRAIN_PATH, header=None, names=COLUMNS)
print('Shape:', raw.shape)
raw[['protocol_type','service','flag','src_bytes','dst_bytes','count','label']].head()

In [ ]:
# Binary label distribution: normal vs attack
labels = (raw['label'] != 'normal').map({False: 'BENIGN', True: 'ATTACK'})
print(labels.value_counts())
print('\nAttack ratio:', round((raw['label'] != 'normal').mean(), 4))

## 2. Why log-transform?
Features like `src_bytes` span 0 → millions and are extremely right-skewed.
A `log1p` transform compresses that range so scaling becomes meaningful — this
is what lifts KDDTest+ accuracy from ~76% to ~84%.

In [ ]:
print('Skewed features that get log1p:', SKEWED)
print('\nBefore log1p  -> src_bytes max:', raw['src_bytes'].max())
print('After  log1p  -> src_bytes max:', round(float(np.log1p(raw['src_bytes']).max()), 3))

## 3. Full preprocessing
`load_dataset` applies the label binarization + log1p transform. One-hot encoding
and scaling are then fit on the training data (and reused at inference).

In [ ]:
from sklearn.preprocessing import StandardScaler

X_train_raw, y_train = load_dataset(TRAIN_PATH)
X_test_raw,  y_test  = load_dataset(TEST_PATH)

# Encode train+test together so columns align (test has unseen services)
combined = pd.concat([X_train_raw, X_test_raw], keys=['train','test'])
combined = pd.get_dummies(combined, columns=CATEGORICAL)
X_train_df, X_test_df = combined.xs('train'), combined.xs('test')

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_df.values.astype('float32'))
X_test  = scaler.transform(X_test_df.values.astype('float32'))

print('X_train:', X_train.shape, '| X_test:', X_test.shape)
print('Total features after one-hot encoding:', X_train.shape[1])

The processed arrays (`X_train`, `X_test`, `y_train`, `y_test`) are now ready for
the model. See **02_model_training** for the CNN-LSTM, or just run
`python scripts/model_trainer.py` to train + evaluate end-to-end.